In [0]:
CREATE TABLE IF NOT EXISTS ecommerce_lakehouse.audit.data_quality_results
(
    table_name STRING,
    check_name STRING,
    failed_records BIGINT,
    total_records BIGINT,
    status STRING,
    execution_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
%python
from pyspark.sql.functions import (
    count,
    sum,
    when,
    current_timestamp,
    col,
    lit
)

In [0]:
%python
def run_quality_check(
    table_name,
    check_name,
    failed_records,
    total_records
):

    status = "PASS" if failed_records == 0 else "FAIL"

    result = spark.createDataFrame(
        [
            (
                table_name,
                check_name,
                int(failed_records),
                int(total_records),
                status
            )
        ],
        [
            "table_name",
            "check_name",
            "failed_records",
            "total_records",
            "status"
        ]
    )

    (
        result
        .withColumn(
            "execution_timestamp",
            current_timestamp()
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "ecommerce_lakehouse.audit.data_quality_results"
        )
    )

In [0]:
%python
df = spark.table(
    "ecommerce_lakehouse.silver.customers"
)

total = df.count()

failed = df.filter(
    col("customer_id").isNull()
).count()

run_quality_check(
    "customers",
    "customer_id_not_null",
    failed,
    total
)

In [0]:
%python
df = spark.table(
    "ecommerce_lakehouse.silver.orders"
)

total = df.count()

failed = df.filter(
    col("order_id").isNull()
).count()

run_quality_check(
    "orders",
    "order_id_not_null",
    failed,
    total
)

In [0]:
%python
orders = spark.table(
    "ecommerce_lakehouse.silver.orders"
)

customers = spark.table(
    "ecommerce_lakehouse.silver.customers"
)

invalid_orders = (
    orders
    .join(
        customers.select("customer_id"),
        "customer_id",
        "left_anti"
    )
)

failed = invalid_orders.count()

total = orders.count()

run_quality_check(
    "orders",
    "customer_foreign_key",
    failed,
    total
)

In [0]:
%python
orders = spark.table(
    "ecommerce_lakehouse.silver.orders"
)

customers = spark.table(
    "ecommerce_lakehouse.silver.customers"
)

invalid_orders = (
    orders
    .join(
        customers.select("customer_id"),
        "customer_id",
        "left_anti"
    )
)

failed = invalid_orders.count()

total = orders.count()

run_quality_check(
    "orders",
    "customer_foreign_key",
    failed,
    total
)

In [0]:
%python
from pyspark.sql.functions import col

order_items = spark.table(
    "ecommerce_lakehouse.silver.order_items"
)

total = order_items.count()

failed = order_items.filter(
    col("price") < 0
).count()

run_quality_check(
    "order_items",
    "price_non_negative",
    failed,
    total
)

In [0]:
%python
reviews = spark.table(
    "ecommerce_lakehouse.silver.reviews"
)

total = reviews.count()

failed = reviews.filter(
    (col("review_score") < 1) |
    (col("review_score") > 5)
).count()

run_quality_check(
    "reviews",
    "review_score_between_1_and_5",
    failed,
    total
)

In [0]:
SELECT *
FROM ecommerce_lakehouse.audit.data_quality_results
ORDER BY execution_timestamp DESC;

table_name,check_name,failed_records,total_records,status,execution_timestamp
reviews,review_score_between_1_and_5,1,102941,FAIL,2026-11-01T11:55:08.065Z
order_items,price_non_negative,0,112650,PASS,2026-11-01T11:55:02.567Z
orders,customer_foreign_key,0,99441,PASS,2026-11-01T11:54:55.552Z
orders,customer_foreign_key,0,99441,PASS,2026-11-01T11:54:49.425Z
orders,order_id_not_null,0,99441,PASS,2026-11-01T11:54:43.678Z
customers,customer_id_not_null,0,99441,PASS,2026-11-01T11:54:38.418Z
reviews,review_score_between_1_and_5,1,102941,FAIL,2026-11-01T11:39:29.972Z
order_items,price_non_negative,0,112650,PASS,2026-11-01T11:35:08.288Z
reviews,review_score_between_1_and_5,1,102941,FAIL,2026-10-22T15:10:27.658Z
order_items,price_non_negative,0,112650,PASS,2026-10-22T15:10:07.230Z
